# Урок 11 - Протокол агент-агент (A2A)


## Установка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Что такое протокол A2A?

**Протокол Агент-агент (A2A)** — это открытый стандарт, который позволяет агентам ИИ общаться,
обнаруживать друг друга и сотрудничать — даже если они построены на разных платформах или размещены
на разных сервисах.

Ключевые концепции:

- **Обнаружение** – Агенты публикуют *Карту агента*, которая описывает их возможности, что упрощает
  другим агентам (или оркестраторам) найти нужного специалиста для задачи.
- **Передача сообщений** – Агенты обмениваются структурированными сообщениями через общий протокол,
  так что запрос от одного агента может быть понят и выполнен другим независимо от его внутренней
  реализации.
- **Жизненный цикл задачи** – A2A определяет состояния такие как *отправлено*, *в работе*, *завершено* и
  *сбой*, давая оркестратору полный контроль над ходом выполнения делегированной задачи.

В этом уроке мы моделируем сотрудничество в стиле A2A, связав три специализированных турагентов
в рабочий процесс, где каждый агент вносит свою экспертизу и передает результаты следующему.


## Создание специализированных туристических агентов


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Совместная работа нескольких агентов через рабочий процесс

Мы связываем трех агентов в последовательный рабочий процесс, который отражает передачу сообщений A2A:

1. **CurrencyExchangeAgent** получает запрос пользователя и создает рекомендации по валюте.
2. **ActivityPlannerAgent** получает расширенный контекст и добавляет рекомендации по мероприятиям.
3. **TravelManagerAgent** синтезирует оба ввода в итоговое туристическое резюме.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Понимание A2A в производственной среде

В производственной среде протокол A2A открывает мощные сценарии межсервисного взаимодействия:

| Возможности | Описание |
|---|---|
| **Межфреймворковая совместимость** | Агент, созданный на одном фреймворке, может делегировать задачи агенту, построенному на любом другом совместимом с A2A фреймворке, что обеспечивает настоящее межорганизационное взаимодействие. |
| **Границы сервисов** | Агенты могут находиться в отдельных микросервисах, облачных регионах или даже в разных организациях, при этом беспрепятственно сотрудничая. |
| **Динамическое обнаружение** | Оркестратор может обращаться к реестру Agent Card во время выполнения, чтобы найти наиболее подходящего специалиста для конкретной подзадачи. |
| **Потоковая передача и push-уведомления** | A2A поддерживает Server-Sent Events (SSE) для обновлений прогресса в реальном времени и push-уведомления для длительных задач. |

Построенный нами выше рабочий процесс является упрощенной, встроенной версией этого шаблона. В реальном
развертывании каждый агент будет предоставлять HTTP-эндпоинт, публиковать Agent Card и обмениваться сообщениями
через протокол A2A JSON-RPC.


## Итог

В этом уроке вы узнали:

1. **Что такое протокол A2A** — открытый стандарт для обнаружения, обмена сообщениями
   и управления задачами между агентами.
2. **Как создавать специализированных агентов** — агента обмена валют, агента планирования
   активности и оркестратора управления путешествиями.
3. **Как подключать агентов в рабочий процесс** — используя `WorkflowBuilder` для моделирования
   последовательной передачи сообщений между агентами.
4. **Как A2A работает в продакшене** — обеспечивая кросс-фреймворковое, кросс-сервисное
   взаимодействие с динамическим обнаружением и потоковыми обновлениями.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
